In [2]:
import os
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module ='src.utils.train_utils')
import json
import matplotlib
matplotlib.use("Agg")
warnings.filterwarnings("ignore", category=matplotlib.MatplotlibDeprecationWarning)
import pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from imblearn.over_sampling import SMOTE
from sklearn import metrics
from imblearn.pipeline import Pipeline as ImbPipeline
from boruta import BorutaPy
import shap

warnings.filterwarnings("ignore", message = "The NumPy global RNG was seeded", category = FutureWarning,)

In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
import os


def load_uci_data(filepath="data/pd_speech_features.csv"):
    """
    Loads the UCI Parkinson's disease speech features dataset.

    The dataset contains multiple voice recordings per patient. This function 
    handles skipping header descriptors and separates the features, 
    labels, and patient identifiers.

    Args:
        filepath (str): The relative or absolute path to the CSV dataset.

    Returns:
        tuple: (X_features, y, groups) where:
            - X_features (pd.DataFrame): The input features (excluding ID and class).
            - y (pd.Series): The binary target labels (0=Healthy, 1=Parkinson's).
            - groups (pd.Series): The patient IDs (to prevent data leakage).
    """
    if not os.path.exists(filepath):
        # Handle relative path if run from src
        filepath = os.path.join("..", filepath)

    df = pd.read_csv(filepath, header=1)

    # X: Features (all columns except 'id' and 'class')
    # y: Target ('class')
    # groups: Patient ID ('id')

    X = df.drop(columns=["class"])
    y = df["class"]
    groups = df["id"]

    # We should also keep version of X that doesn't have 'id' for model training
    X_features = X.drop(columns=["id"])

    return X_features, y, groups


def get_train_test_split(X, y, groups, test_size=0.2, random_state=42):
    """
    Splits the data into training and holdout sets while ensuring
    patient IDs (groups) are never shared between sets.

    Args:
        X (pd.DataFrame): The feature matrix.
        y (pd.Series): The target labels.
        groups (pd.Series): Patient IDs for group-based splitting.
        test_size (float, optional): Proportion of patients for the test set. Defaults to 0.2.
        random_state (int, optional): Random seed for reproducibility. Defaults to 42.

    Returns:
        tuple: (X_train, X_test, y_train, y_test, groups_train, groups_test)
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    return (
        X.iloc[train_idx],
        X.iloc[test_idx],
        y.iloc[train_idx],
        y.iloc[test_idx],
        groups.iloc[train_idx],
        groups.iloc[test_idx],
    )


def get_stratified_group_kfold(n_splits=5, shuffle=True, random_state=42):
    """
    Creates a StratifiedGroupKFold cross-validator for leakage-proof validation.

    This ensures that the same patient ID does not appear in both train and test sets,
    while maintaining the distribution of target classes in each fold.

    Args:
        n_splits (int, optional): Number of folds. Defaults to 5.
        shuffle (bool, optional): Whether to shuffle the data. Defaults to True.
        random_state (int, optional): Random seed for reproducibility. Defaults to 42.

    Returns:
        StratifiedGroupKFold: The configured scikit-learn cross-validator.
    """
    return StratifiedGroupKFold(
        n_splits=n_splits, shuffle=shuffle, random_state=random_state
    )


if __name__ == "__main__":
    X, y, groups = load_uci_data(
        "C:/Users/seahy/OneDrive - Singapore Management University/Term 2/CS610 Applied Machine Learning/Project/Data/pd_speech_features.csv"
    )
    print(f"Data ingested: {X.shape[0]} samples, {X.shape[1]} features.")
    print(f"Target distribution:\n{y.value_counts(normalize=True)}")
    print(f"Unique Patient IDs: {groups.nunique()}")


import pandas as pd
import numpy as np
import json
import os
import time
import random
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    accuracy_score,
    f1_score,
    roc_curve,
    log_loss
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
#from .data_loader import get_stratified_group_kfold


def set_seed(seed=42):
    """
    Sets global seeds for reproducibility across numpy, python random, and sklearn.

    Args:
        seed (int, optional): The random seed value. Defaults to 42.

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Global seed set to: {seed}")


def get_preprocessing_pipeline(dim_reduction="pca", n_components=0.95):
    """
    Standardized preprocessing pipeline.
    Scaling + Dimensionality Reduction (PCA or none).

    Args:
        dim_reduction (str, optional): Type of reduction ("pca" or None). Defaults to "pca".
        n_components (float/int, optional): Variance explained or number of components. Defaults to 0.95.

    Returns:
        Pipeline: The configured scikit-learn Pipeline.
    """
    steps = [("scaler", StandardScaler())]

    if dim_reduction == "pca":
        steps.append(("pca", PCA(n_components=n_components)))

    return Pipeline(steps)


def train_with_random_search(
    estimator,
    param_distributions,
    X,
    y,
    groups,
    n_iter=50,
    cv_splits=5,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1,
):
    """
    Wrapper for RandomizedSearchCV using StratifiedGroupKFold.

    Args:
        estimator (BaseEstimator): The model or pipeline to tune.
        param_distributions (dict): The hyperparameter search space.
        X (pd.DataFrame): Training features.
        y (pd.Series): Training labels.
        groups (pd.Series): Patient IDs for grouped cross-validation.
        n_iter (int, optional): Number of parameter settings sampled. Defaults to 50.
        cv_splits (int, optional): Number of cross-validation folds. Defaults to 5.
        scoring (str, optional): Metric to optimize. Defaults to "f1_weighted".
        random_state (int, optional): Seed for sampling. Defaults to 42.
        n_jobs (int, optional): Number of parallel jobs. Defaults to -1.

    Returns:
        tuple: (search, fit_time) where search is the fitted RandomizedSearchCV object.
    """
    cv = get_stratified_group_kfold(n_splits=cv_splits, random_state=random_state)

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        random_state=random_state,
        n_jobs=n_jobs,
        verbose=1,
        return_train_score=True,
    )

    start_time = time.time()
    search.fit(X, y, groups=groups)
    end_time = time.time()

    fit_time = end_time - start_time

    return search, fit_time


def log_results(
    model_name, search_results, fit_time, log_path="outputs/results_log.csv"
):
    """
    Logs the best parameters, mean F1 score, and fit time to a shared CSV file.

    Args:
        model_name (str): Label for the model being logged.
        search_results (RandomizedSearchCV): The fitted search object.
        fit_time (float): Time taken for the search in seconds.
        log_path (str, optional): Path to the output CSV. Defaults to "outputs/results_log.csv".

    Returns:
        None
    """
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    best_params = search_results.best_params_
    best_score = search_results.best_score_

    # Handle numpy types that are not JSON serializable
    def json_default(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return str(obj)

    log_entry = {
        "model_name": model_name,
        "best_params": json.dumps(best_params, default=json_default),
        "cv_mean_f1_weighted": best_score,
        "fit_time_seconds": round(fit_time, 2),
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    df_new = pd.DataFrame([log_entry])

    if os.path.exists(log_path):
        df_existing = pd.read_csv(log_path)
        df_final = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_final = df_new

    df_final.to_csv(log_path, index=False)
    print(f"Results logged to {log_path}")


def save_model_packet(model_name, search_results, test_auc=None, output_dir="outputs"):
    """
    Saves results.json as per the 'Model Packet' requirement.

    Args:
        model_name (str): Label for the model.
        search_results (RandomizedSearchCV): The fitted search object.
        test_auc (float, optional): AUC score on the holdout set. Defaults to None.
        output_dir (str, optional): Target directory. Defaults to "outputs".

    Returns:
        None
    """
    os.makedirs(output_dir, exist_ok=True)

    results = {
        "model_name": model_name,
        "best_params": search_results.best_params_,
        "cv_accuracy_mean": None,  # Should be collected if accuracy scoring was used
        "cv_f1_mean": search_results.best_score_,
        "test_auc": test_auc,
    }

    # Try to find accuracy if multiple metrics were used
    if isinstance(search_results.cv_results_, dict):
        if "mean_test_accuracy" in search_results.cv_results_:
            results["cv_accuracy_mean"] = search_results.cv_results_[
                "mean_test_accuracy"
            ][search_results.best_index_]

    file_path = os.path.join(output_dir, f"results_{model_name.lower()}.json")

    # Pre-process results to convert numpy types for JSON compatibility
    def convert_numpy(obj):
        if isinstance(obj, dict):
            return {k: convert_numpy(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [convert_numpy(i) for i in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj

    clean_results = convert_numpy(results)

    with open(file_path, "w") as f:
        json.dump(clean_results, f, indent=4)

    print(f"Model packet results saved to {file_path}")


def save_model(model, model_name, output_dir="models"):
    """
    Saves a trained model to a .joblib file.

    Args:
        model (BaseEstimator): The trained model object.
        model_name (str): Label for the filename.
        output_dir (str, optional): Target directory. Defaults to "models".

    Returns:
        None
    """
    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{model_name.lower()}.joblib")
    joblib.dump(model, file_path)
    print(f"Model saved to {file_path}")


def plot_confusion_matrix(y_true, y_pred, model_name):
    """
    Standardized plotter for Confusion Matrix.

    Args:
        y_true (np.array/pd.Series): Correct labels.
        y_pred (np.array/pd.Series): Predicted labels.
        model_name (str): Model name for the plot title.

    Returns:
        None
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()


def get_optimal_threshold(y_true, y_prob):
    """
    Finds the optimal threshold that maximizes the Geometric Mean (G-Mean).
    G-Mean = sqrt(Sensitivity * Specificity)

    Args:
        y_true (np.array/pd.Series): Correct labels.
        y_prob (np.array): Predicted probabilities for the positive class.

    Returns:
        float: The optimal classification threshold.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    gmeans = np.sqrt(tpr * (1 - fpr))
    ix = np.argmax(gmeans)
    return thresholds[ix]


def get_threshold_at_recall(y_true, y_prob, target_recall=0.95):
    """
    Finds the threshold that achieves a specific recall (Sensitivity).
    Useful for screening tools where missing a case is critical.

    Args:
        y_true (np.array/pd.Series): Correct labels.
        y_prob (np.array): Predicted probabilities for the positive class.
        target_recall (float, optional): Desired recall level. Defaults to 0.95.

    Returns:
        float: The threshold required to meet the target recall.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    # Find the threshold where recall (tpr) is >= target_recall
    ix = np.where(tpr >= target_recall)[0][0]
    return thresholds[ix]


def evaluate_model(model, X_train, y_train, X_test, y_test, model_name, threshold=0.5):
    """
    Standardized evaluator for the holdout set.
    Supports a custom classification threshold.

    Args:
        model (BaseEstimator): The trained model to evaluate.
        X_train (pd.DataFrame): Training features (for gap analysis).
        y_train (pd.Series): Training labels (for gap analysis).
        X_test (pd.DataFrame): Test features.
        y_test (pd.Series): Test labels.
        model_name (str): Label for output displays.
        threshold (float, optional): Classification threshold. Defaults to 0.5.

    Returns:
        dict: A dictionary containing Accuracy, F1, AUC, and Threshold metrics.
    """
    # Test Metrics
    y_prob_test = (
        model.predict_proba(X_test)[:, 1]
        if hasattr(model, "predict_proba")
        else model.decision_function(X_test)
    )
    y_pred_test = (y_prob_test >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test, average="macro")
    auc = roc_auc_score(y_test, y_prob_test)

    # Train Metrics
    y_prob_train = (
        model.predict_proba(X_train)[:, 1]
        if hasattr(model, "predict_proba")
        else model.decision_function(X_train)
    )
    y_pred_train = (y_prob_train >= threshold).astype(int)

    train_acc = accuracy_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train, average="macro")
    train_auc = roc_auc_score(y_train, y_pred_train)
    train_recall = recall_score(y_train, y_pred_train)

    gap = (train_f1 - f1) / train_f1 * 100 if train_f1 > 0 else 0

    aucgap = (train_auc - auc) / train_auc * 100 if train_auc > 0 else 0

    print(f"\n--- {model_name} Evaluation ---")
    if threshold != 0.5:
        print(f"Using Optimized Threshold: {threshold:.4f}")

    print(f"\n[Overfitting Check]")
    print(f"Train F1: {train_f1:.4f} | Test F1: {f1:.4f} | Gap: {gap:+.2f}%")

    if gap > 10:
        print(
            ">> WARNING: High overfitting gap detected (>10%). Model may not generalize well."
        )
    else:
        print(">> OK: Overfitting gap is within acceptable limits.")
    
    print(f"Train AUC: {train_auc:.4f} | Test AUC: {auc:.4f} | Gap: {aucgap:+.2f}%")

    if aucgap > 10:
        print(
            ">> WARNING: High overfitting gap detected (>10%). Model may not generalize well."
        )
    else:
        print(">> OK: Overfitting gap is within acceptable limits.")

    trainloss = log_loss(y_train, y_prob_train)

    print(f"Log Loss (Train): {trainloss:.4f}")

    testloss = log_loss(y_test, y_prob_test)

    print(f"Log Loss (Holdout): {testloss:.4f}")

    # Check for OOB Score (specific to RandomForest)
    if hasattr(model, "estimator") and hasattr(model.estimator, "named_steps"):
        clf_step = model.estimator.named_steps.get("clf")
        if clf_step and hasattr(clf_step, "oob_score_"):
            print(f"OOB Accuracy Score: {clf_step.oob_score_:.4f}")



    print(f"\n[Holdout Test Results]")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC Score: {auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_test, y_pred_test))

    plot_confusion_matrix(y_test, y_pred_test, model_name)

    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc,
        "threshold": threshold,
        "train_f1": train_f1,
        "gap": gap,
    }


def aggregate_patient_data(X, y, groups):
    """
    Averages multiple samples per patient and adds variability features (std, skew).
    Each patient in this dataset has 3 recordings; calculating the mean and
    variability (std) captures both the average state and voice instability.
    """
    df = X.copy()
    df["target"] = y
    df["patient_id"] = groups

    # Define aggregation: mean, std, skew for features
    agg_funcs = ["mean", "std", "skew"]

    # Apply aggregations
    df_agg = df.groupby("patient_id").agg(
        {**{col: agg_funcs for col in X.columns}, "target": "first"}
    )

    # Flatten multi-index columns
    new_cols = []
    for col in df_agg.columns:
        if col[0] == "target":
            new_cols.append("target")
        else:
            new_cols.append(f"{col[0]}_{col[1]}")
    df_agg.columns = new_cols
    df_agg = df_agg.reset_index()

    # Drop rows with NaN if any (though unlikely for 3 samples)
    df_agg = df_agg.fillna(0)

    X_agg = df_agg.drop(columns=["patient_id", "target"])
    y_agg = df_agg["target"]
    groups_agg = df_agg["patient_id"]

    return X_agg, y_agg, groups_agg


def evaluate_with_voting(y_true, y_prob, groups, threshold=0.5):
    """
    Aggregates record-level predictions into patient-level predictions via soft voting (averaging probabilities).
    This mimics a majority vote but is more robust for probabilistic models.

    Args:
        y_true (pd.Series/np.array): True labels per record.
        y_prob (np.array): Predicted probabilities per record.
        groups (pd.Series): Patient IDs for each record.
        threshold (float): Threshold to convert averaged probability to labels.

    Returns:
        dict: Patient-level metrics.
    """
    df = pd.DataFrame(
        {"patient_id": groups, "y_true": y_true, "y_prob": y_prob}
    )

    # Group by patient and average probabilities
    patient_results = df.groupby("patient_id").agg(
        {"y_true": "first", "y_prob": "mean"}
    )

    y_true_patient = patient_results["y_true"]
    y_prob_patient = patient_results["y_prob"]
    y_pred_patient = (y_prob_patient >= threshold).astype(int)

    acc = accuracy_score(y_true_patient, y_pred_patient)
    f1 = f1_score(y_true_patient, y_pred_patient, average="macro")
    rec = recall_score(y_true_patient, y_pred_patient, average='binary')

    votingloss = log_loss(y_true_patient, y_prob_patient)

    print(f"Log Loss (Voting): {votingloss:.4f}")

    # Handle cases where only one class is present in a tiny test set split (unlikely but possible)
    try:
        auc = roc_auc_score(y_true_patient, y_prob_patient)
    except ValueError:
        auc = 0.0

    return {
        "recall": rec,
        "accuracy": acc,
        "f1": f1,
        "auc": auc,
        "y_true": y_true_patient,
        "y_pred": y_pred_patient,
        "y_prob": y_prob_patient,
        "votingloss": votingloss
    }

from scipy.stats import uniform
from functools import partial

def run_xgb_model(config_path = None, use_voting = False ):
    if config_path is None or config_path == "-f":
        config_path = r"C:/Users/seahy/OneDrive - Singapore Management University/Term 2/CS610 Applied Machine Learning/Project/configs/xgboost_config_boruta_pred_agg.json"
    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config=json.load(f)
        raw_grid = config['param_grid']
        cleaned_grid = {}
        for param, value in raw_grid.items():
            if isinstance(value, dict) and value.get("distribution") == "uniform":
        # Scipy uniform takes (start, width), so: loc=min, scale=max-min
                cleaned_grid[param] = uniform(value["min"], value["max"] - value["min"])
            else:
        # Keep lists as they are (e.g., n_estimators: [100, 200])
                cleaned_grid[param] = value
                print(f"Loaded configuration from {config_path}")
    else:
        print(f"Config not found at {config_path}, using defaults.")
        config = {
            "model_name": "xgb_model",
            "data_path": "data/pd_speech_features.csv",
            "test_size": 0.2,
            "random_state": 42,
            "n_iter": 10,
            "param_grid": {"clf__n_estimators":[100,200]},
          }
        
    set_seed(config.get("random_state", 42))
    MODEL_NAME = config.get("model_name", "xgb_model")
    DATA_PATH = config.get("data_path", "data/pd_speech_features.csv")
    
    print(f"--- Starting {MODEL_NAME} ---")
    
    X, y, groups = load_uci_data(DATA_PATH)

      # Optional Step 1: Patient-Level Aggregation
    if config.get("aggregate_patients", False):
        print(
            f"Aggregating samples at patient level... (Original scale: {len(X)} samples)"
        )
        X, y, groups = aggregate_patient_data(X, y, groups)
        print(f"New scale: {len(X)} patient profiles.")
    
    # Patient-grouped holdout split (20% test)
    X_train, X_test, y_train, y_test, groups_train, groups_test = get_train_test_split(
        X,
        y,
        groups,
        test_size=config.get("test_size", 0.2),
        random_state=config.get("random_state", 42),
    )
    
    print(f"Training set: {len(X_train)} samples ({groups_train.nunique()} patients)")
    print(f"Holdout set:  {len(X_test)} samples ({groups_test.nunique()} patients)")
    
    # 3. Pipeline Setup
    # Flatten the pipeline structure: imblearn Pipeline doesn't like nested Pipelines for intermediate steps
    pre_steps = get_preprocessing_pipeline(dim_reduction=None).steps
    
    # Selection method from config
    fs_method = config.get("feature_selection", {}).get("method", "f_classif")
    print(f"Using feature selection method: {fs_method}")
    
    if fs_method == "boruta":
        print("\n--- Configuring Boruta Feature Selection in Pipeline ---")
        # Boruta needs a basic RF; using n_jobs=None to avoid joblib UserWarning
        rf_boruta = RandomForestClassifier(
            n_estimators = 'auto',
            max_depth = 5,
            n_jobs= None,
            random_state=config.get("random_state", 42),
        )
    
        boruta_selector = BorutaPy(
            rf_boruta,
            n_estimators="auto",
            verbose=0,
            random_state=config.get("random_state", 42),
            max_iter=100,  # Can increase for more stability
        )
    
        original_columns_for_visuals = X.columns
    
        os.makedirs("outputs/pipeline_cache", exist_ok=True)
        # Simplify pipeline since features are already selected
        full_pipeline = ImbPipeline(
            steps=pre_steps
            + [
                ("boruta", boruta_selector),
                ("smote", SMOTE(random_state=config.get("random_state", 42))),
                (
                    "clf",
                    XGBClassifier(
                        random_state=config.get("random_state", 42),
                        objective = 'binary:logistic',
                        booster = 'gbtree',
                        eval_metric = 'auc'
                    ),
                ),
            ],
            memory="outputs/pipeline_cache",
        )

    else:
        # Use partial to 'freeze' the random_state for mutual_info_classif
        if fs_method == "mutual_info":
            score_func = partial(mutual_info_classif, random_state=config.get("random_state", 42))
        else:
            score_func = f_classif # f_classif is deterministic (purely statistical)
    
        original_columns_for_visuals = X.columns
        full_pipeline = ImbPipeline(
            steps=pre_steps
            + [
                ("selector", SelectKBest(score_func=score_func, k=100)),
                ("smote", SMOTE(random_state=config.get("random_state", 42))),
                (
                    "clf",
                    XGBClassifier(
                        random_state=config.get("random_state", 42),
                        objective='binary:logistic',
                        booster='gbtree',
                        eval_metric='auc',
                        n_jobs=1  # Highly recommended for consistency
                    ),
                ),
            ]
        )
    # 4. Hyperparameter Search Strategy
    #raw_params = config.get("param_grid", {})
    #param_dist = {}
    #for key, value in raw_params.items():
      #if fs_method == "boruta" and "selector__" in key:
        #continue
      #param_dist[key] = value
    
    tuning_metric = config.get("tuning_metric", "f1_weighted")
    print(
          f"\nRunning Randomized Search CV ({config.get('n_iter', 50)} iterations) using {tuning_metric})..."
    )
    search, fit_time = train_with_random_search(
      estimator = full_pipeline,
      param_distributions = cleaned_grid,
      X= X_train,
      y=y_train,
      groups = groups_train,
      n_iter=config.get("n_iter", 50),
      scoring = tuning_metric,
      random_state = config.get("random_state", 42),
    )
    tuning_metric_name = str(tuning_metric).capitalize()
    print(f"Best CV {tuning_metric_name} Score: {search.best_score_:.4f}")
    
    # 5. Probability Calibration
    # Refines probabilities to be more representative of real underlying risk
    print("\nCalibrating Probabilities...")
    cv_splits = list(
      get_stratified_group_kfold(
        n_splits = 5, shuffle = True, random_state = config.get("random_state", 42)
      ).split(X_train, y_train, groups = groups_train)
    )
    calibrated_model = CalibratedClassifierCV(
      search.best_estimator_, method = "sigmoid", cv=cv_splits
    )
    calibrated_model.fit(X_train,y_train)
    
    print("Optimizing decision threshold (on calibrated probabilities)...")
    y_probs_train = calibrated_model.predict_proba(X_train)[:, 1]
    
    target_recall_val = config.get("target_recall", None)
    if target_recall_val is not None:
      target_recall_val = float(target_recall_val)
      print(f"Targeting {target_recall_val*100:.1f}% Sick Recall (Sensitivity)...")
      opt_threshold = get_threshold_at_recall(
        y_train, y_probs_train, target_recall = target_recall_val
      )
    else:
      opt_threshold = get_optimal_threshold(y_train, y_probs_train)
    
    test_metrics = evaluate_model(
      calibrated_model,
      X_train,
      y_train,
      X_test,
      y_test,
      MODEL_NAME,
      threshold=opt_threshold,
    )
        # 6b. Optional: Prediction Aggregation (Majority Vote)
    if use_voting:
        print("\n--- Patient-Level Evaluation (Majority Vote) ---")
        y_probs_test = calibrated_model.predict_proba(X_test)[:, 1]
        voting_results = evaluate_with_voting(y_test, y_probs_test, groups_test, threshold=opt_threshold)
        print(f"Voted Recall: {voting_results['recall']:.4f}")
        print(f"Voted Accuracy: {voting_results['accuracy']:.4f}")
        print(f"Voted F1 Score: {voting_results['f1']:.4f}")
        print(f"Voted AUC Score: {voting_results['auc']:.4f}")
        print(f"Voted Logloss: {voting_results['votingloss']:.4f}")
        
        # Add to test_metrics for possible later use (though not currently logged)
        test_metrics['voted_f1'] = voting_results['f1']
        test_metrics['voted_auc'] = voting_results['auc']
        
    log_results(MODEL_NAME, search, fit_time, log_path='outputs/results_log.csv')
    
    search.best_params_["optimal_threshold"] = opt_threshold
    save_model_packet(
      MODEL_NAME, search, test_auc = test_metrics["auc"], output_dir="outputs"
    )
    print("\n--- Comparing Probability Strategies ---")
    
    y_prob_initial = search.best_estimator_.predict_proba(X_test)[:, 1]
    
    y_prob_calibrated = calibrated_model.predict_proba(X_test)[:, 1]
    
    prob_comparison = pd.DataFrame({
        "Patient_ID": groups_test.values,
        "True_Class": y_test.values,
        "Raw_XGB_Prob": y_prob_initial,
        "Calibrated_Prob": y_prob_calibrated
    })
    
    # 3. Patient-Level Strategy (Averaged across the 3 recordings)
    # We group the calibrated probabilities by patient ID
    patient_level_probs = prob_comparison.groupby("Patient_ID")["Calibrated_Prob"].transform("mean")
    prob_comparison["Patient_Voted_Prob"] = patient_level_probs
    
    print(prob_comparison.head(10))

    prob_comparison = pd.DataFrame({
        "Patient_ID": groups_test.values,
        "True_Class": y_test.values,
        "Raw_XGB_Prob": search.best_estimator_.predict_proba(X_test)[:, 1],
        "Calibrated_Prob": calibrated_model.predict_proba(X_test)[:, 1]
    })
    
    # Calculate Patient-Level Voted Probability
    prob_comparison["Patient_Voted_Prob"] = prob_comparison.groupby("Patient_ID")["Calibrated_Prob"].transform("mean")
    
    prob_comparison.to_csv("outputs/probability_comparison.csv", index=False)

    save_model(calibrated_model, MODEL_NAME, output_dir = "models")
    generate_visuals(
            calibrated_model,
            search.best_estimator_,
            X_train,
            y_train,
            groups_train,
            original_columns_for_visuals,
            MODEL_NAME,
            fs_method
    )

    return prob_comparison
    
def generate_visuals(
    best_model,
    base_pipeline,
    X_train,
    y_train,
    groups_train,
    feature_names,
    model_name,
    fs_method,
):
    print("\nGenerating Visuals...")
    os.makedirs("outputs/plots", exist_ok = True)
    if hasattr(best_model, "calibrated_classifiers_"):
        clf = best_model.calibrated_classifiers_[0].estimator.named_steps["clf"]
        pipeline_head=best_model.calibrated_classifiers_[0].estimator
    else:
        clf = best_model.named_steps["clf"]
        pipeline_head = best_model
    if fs_method == "boruta":
        selector = pipeline_head.named_steps["boruta"]
        mask = selector.support_
    else:
        selector = pipeline_head.named_steps["selector"]
        mask = selector.get_support()
    selected_features = feature_names[mask]

    importances = clf.feature_importances_
    indices = np.argsort(importances)[-10:]

    plt.figure(figsize=(12,6))
    plt.title(f"Top 10 Physical Feature Importance ({model_name})")
    plt.barh(range(len(indices)), importances[indices], color = 'teal', align ='center')
    plt.yticks(range(len(indices)), [selected_features[i] for i in indices])
    plt.xlabel("Gini Importance (on selected features)")
    plt.tight_layout()
    plt.savefig("outputs/plots/xbg_feature_importance.png")

    print(
        f"Top physical features found {', '.join([selected_features[i] for i in indices[-1:]])}"
    )

    print("Generating Validation Curve for n_estimators...")
    param_range = [100, 200, 500, 800 ,1000]

    train_scores, test_scores = validation_curve(
        base_pipeline,
        X_train,
        y_train,
        param_name = "clf__n_estimators",
        param_range=param_range,
        cv=5,
        scoring = "f1_weighted",
        n_jobs = -1,
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis = 1)
    test_mean = np.mean(test_scores, axis = 1)
    test_std = np.std(test_scores, axis =1)

    plt.figure(figsize = (10,6))
    plt.plot(
        param_range, train_mean, label='Training Score', color = "darkorange", marker ='o'
    )
    plt.fill_between(
        param_range,
        train_mean - train_std,
        train_mean + train_std,
        alpha = 0.2,
        color = "darkorange"
    )
    plt.plot(
        param_range, test_mean, label = "Cross-validation score", color ='navy', marker ='s'
    )
    plt.fill_between(
        param_range, test_mean - test_std, test_mean + test_std, alpha = 0.2, color = 'navy'
    )

    plt.title(f"Validation curve with {model_name} (n_estimators)")
    plt.xlabel("Number of Estimators")
    plt.ylabel("F1 Weighted Score")
    plt.legend(loc="best")
    plt.grid()
    plt.tight_layout()
    plt.savefig("outputs/plots/xbg_validation_curve.png")

    print(f"Generating SHAP Summary Plot...")

    if fs_method == "boruta":
        X_train_transformed = pipeline_head.named_steps["boruta"].transform(
            X_train.to_numpy()
        )
    else:
        X_train_transformed = pipeline_head.named_steps["selector"].transform(
            X_train.to_numpy()
        )

    try:
        explainer = shap.TreeExplainer(clf)

        shap_values = explainer.shap_values(X_train_transformed)

        if isinstance(shap_values, list):
            shap_values_to_plot = shap_values[1]

        else:
            shap_values_to_plot = shap_values

        X_train_df = pd.DataFrame(X_train_transformed, columns = selected_features)

        plt.figure()
        shap.summary_plot(
            shap_values_to_plot,
            features = X_train_df,
            max_display=10,
            show=False,
            plot_type = "dot"
        )
        plt.title(f"SHAP Summary Plot ({model_name})")
        plt.tight_layout()
        plt.savefig("outputs/plots/xbg_shap_summary.png")
        print("SHAP Summary Plot saved to outputs/plts/xgb_shap_summary.png")
        plt.close()
    except Exception as e:
        print(f"Warning: Could not generate SHAP plot: {e}")

    print("Visuals saved to outputs/plots/")

Data ingested: 756 samples, 753 features.
Target distribution:
class
1    0.746032
0    0.253968
Name: proportion, dtype: float64
Unique Patient IDs: 252


# Milestone 4: XGBoost Group Splitting
This experiment focuses on **Grouped Splitting** while keeping all raw recordings intact (756 rows). Unlike the aggregation approach, this method allows the model to learn from the full variance of the 3 recordings per patient, but requires strict **Patient-Aware Validation** to prevent data leakage.

## Objectives
1. **Grouped Splitting**: Maintain all raw records but split strictly by patient_id (using GroupShuffleSplit and StratifiedGroupKFold).
2. **Boruta**: Identify relevant predictors in the high-variance raw dataset.
3. **Mutual Information**: Measure dependencies between raw features and diagnosis.

# 1. Data Handling: Group Splitting
We keep all 756 records. The critical part is ensuring that samples from the same patient are never split across training and validation sets. get_train_test_split handles this by using patient IDs as groups.

In [4]:
# Perform Aggregation
#X, y, groups = load_uci_data(DATA_PATH)

print(f"Raw Data Scale: {X.shape[0]} recordings | {X.shape[1]} features.")
print(f"Unique Patients: {groups.nunique()}")

# Demonstrate Grouped Split
X_train, X_test, y_train, y_test, g_train, g_test = get_train_test_split(X, y, groups, test_size=0.2)

print("\n--- Patient-Aware Split Results ---")
print(f"Training set: {len(X_train)} samples ({g_train.nunique()} patients)")
print(f"Holdout set:  {len(X_test)} samples ({g_test.nunique()} patients)")

# Check for leakage
overlap = set(g_train).intersection(set(g_test))
print(f"Patient overlap between Train/Test: {len(overlap)} (Expected: 0)")

Raw Data Scale: 756 recordings | 753 features.
Unique Patients: 252

--- Patient-Aware Split Results ---
Training set: 603 samples (201 patients)
Holdout set:  153 samples (51 patients)
Patient overlap between Train/Test: 0 (Expected: 0)


# 2. Feature Selection: Boruta

In [5]:
CONFIG_BORUTA = r"C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json"
run_xgb_model(CONFIG_BORUTA)

Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_boruta_

C:\Users\seahy\AppData\Local\Temp\ipykernel_38932\933818451.py:350: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Model saved to models\my_xgboost_model.joblib
Model saved to models\xgb_model.joblib

Generating Visuals...
Top physical features found std_8th_delta_delta
Generating Validation Curve for n_estimators...
Generating SHAP Summary Plot...
SHAP Summary Plot saved to outputs/plts/xgb_shap_summary.png
Visuals saved to outputs/plots/


# 3. Feature Selection: Mutual Information
Mutual Information (MI) measures the statistical dependence between varaibles. It captures both linear and non-linear relationships.

In [6]:
CONFIG_MI = r"C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json"
run_xgb_model(CONFIG_MI)

Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\configs\xgboost_config_mutual_info_GS.json
Loaded configuration from C:\Users\seahy\OneDrive - Singapore Management University\Term 2\CS610 Applied Machine Learning\Project\confi

C:\Users\seahy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\seahy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\seahy\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\seahy\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Best CV F1_macro Score: 0.6476

Calibrating Probabilities...
Optimizing decision threshold (on calibrated probabilities)...

--- xgb_model Evaluation ---
Using Optimized Threshold: 0.7225

[Overfitting Check]
Train F1: 0.7969 | Test F1: 0.7324 | Gap: +8.10%
>> OK: Overfitting gap is within acceptable limits.
Train AUC: 0.8562 | Test AUC: 0.8754 | Gap: -2.23%
>> OK: Overfitting gap is within acceptable limits.
Log Loss (Train): 0.3484
Log Loss (Holdout): 0.4193

[Holdout Test Results]
Accuracy: 0.7778
F1 Score: 0.7324
AUC Score: 0.8754

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.67      0.62        42
           1       0.87      0.82      0.84       111

    accuracy                           0.78       153
   macro avg       0.73      0.74      0.73       153
weighted avg       0.79      0.78      0.78       153

Results logged to outputs/results_log.csv
Model packet results saved to outputs\results_xgb_model.json


C:\Users\seahy\AppData\Local\Temp\ipykernel_38932\933818451.py:350: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Model saved to models\my_xgboost_model.joblib
Model saved to models\xgb_model.joblib

Generating Visuals...
Top physical features found std_8th_delta_delta
Generating Validation Curve for n_estimators...
Generating SHAP Summary Plot...
SHAP Summary Plot saved to outputs/plts/xgb_shap_summary.png
Visuals saved to outputs/plots/
